# YOLOv8n — CoralSense Colab Training

Use the reviewed Hard Coral / Soft Coral / quadrat_area dataset version. Split by survey/session before train-only augmentation. Run from a fresh GPU runtime.

Configure the version in Step 2. Check validation results to select models; reserve test for final evaluation. Checkpoints and metadata persist to Drive by default. Training budgets/pretraining differ by architecture and must be reported.

Validation is the default evaluation split. Before a long run, use one epoch as a smoke test, inspect boxes and losses, then start a fresh full experiment. Old five-class checkpoints cannot be used in the new comparison.


## Step 1: Install Dependencies

In [ ]:
!pip install -q ultralytics roboflow pyyaml

## Step 2: Download Dataset from Roboflow

**PASTE YOUR ROBOFLOW CODE HERE:**

Replace the code in the next cell with the download code from Roboflow (the code you just copied)

In [ ]:
MODEL_NAME="yolov8n"
from pathlib import Path
from getpass import getpass
from datetime import datetime
import os, json, random, hashlib, importlib.metadata
import numpy as np
import torch
from roboflow import Roboflow

# Fill these from the newly generated Roboflow version. Never paste a key here.
WORKSPACE = ""
PROJECT = ""
DATASET_VERSION = None
EXPECTED_CLASSES = {"Hard Coral", "Soft Coral", "quadrat_area"}
SEED = 42
USE_DRIVE = True
EVAL_SPLIT = "valid"  # Use test only for the final, frozen evaluation.
assert WORKSPACE and PROJECT and isinstance(DATASET_VERSION, int), "Enter workspace, project and version first."
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda": print("GPU:", torch.cuda.get_device_name(0))
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_ROOT = Path('/content/drive/MyDrive/CoralSense')
else:
    OUTPUT_ROOT = Path.cwd() / 'coralsense_outputs'
CHECKPOINT_DIR = OUTPUT_ROOT / 'checkpoints'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR = OUTPUT_ROOT / 'runs' / (datetime.now().strftime('%Y%m%d-%H%M%S-%f') + '-' + MODEL_NAME)
RUN_DIR.mkdir(parents=True)
ROBOFLOW_API_KEY = getpass("Roboflow API key: ")
dataset = Roboflow(api_key=ROBOFLOW_API_KEY).workspace(WORKSPACE).project(PROJECT).version(DATASET_VERSION).download('yolov8')
del ROBOFLOW_API_KEY
dataset_path = Path(dataset.location)

## Step 3: Verify Dataset Structure

In [ ]:
import yaml
from PIL import Image, ImageOps
import torchvision.transforms.functional as TF

with (dataset_path / 'data.yaml').open() as handle:
    data = yaml.safe_load(handle)
names = data['names']
class_names = [names[k] for k in sorted(names, key=int)] if isinstance(names, dict) else list(names)
assert len(class_names) == 3 and set(class_names) == EXPECTED_CLASSES, f"Unexpected classes: {class_names}"
data['names'] = class_names
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.webp'}
def image_paths(directory):
    return sorted(p for p in Path(directory).iterdir() if p.suffix.lower() in IMAGE_EXTENSIONS)

def find_split(dp, split_names):
    for name in split_names:
        aliases = ['valid', 'val'] if name in ('valid', 'val') else [name]
        for split in aliases:
            for images, labels in [(dp / split / 'images', dp / split / 'labels'),
                                   (dp / 'images' / split, dp / 'labels' / split)]:
                if images.is_dir() and labels.is_dir() and image_paths(images): return images, labels
    raise FileNotFoundError(f"Missing/empty split: {split_names}. No silent evaluation fallback.")

def read_annotations(path, label_dir, width, height, offset=0):
    boxes, labels = [], []
    label_path = Path(label_dir) / (Path(path).stem + '.txt')
    if not label_path.exists():
        raise FileNotFoundError(f"Missing label: {label_path.name}. Use an empty file only for reviewed negative images.")
    for line in label_path.read_text().splitlines():
        if not line.strip(): continue
        fields = line.split()
        if len(fields) != 5: raise ValueError(f"Expected detection boxes, not polygons: {label_path.name}")
        cid = int(fields[0]); cx, cy, w, h = map(float, fields[1:])
        if not (0 <= cid < len(class_names) and all(np.isfinite([cx,cy,w,h])) and
                0 <= cx <= 1 and 0 <= cy <= 1 and 0 < w <= 1 and 0 < h <= 1):
            raise ValueError(f"Invalid class/box in {label_path.name}")
        box = [max(0., (cx-w/2)*width), max(0., (cy-h/2)*height),
               min(float(width), (cx+w/2)*width), min(float(height), (cy+h/2)*height)]
        if box[2] <= box[0] or box[3] <= box[1]: raise ValueError(f"Degenerate box: {label_path.name}")
        boxes.append(box); labels.append(cid + offset)
    return {'boxes': torch.tensor(boxes, dtype=torch.float32).reshape(-1,4),
            'labels': torch.tensor(labels, dtype=torch.int64)}

split_dirs = {s: find_split(dataset_path, [s]) for s in ('train','valid','test')}
manifest = {}; seen_hashes = {}; fingerprint = hashlib.sha256()
for split, (images_dir, labels_dir) in split_dirs.items():
    counts = np.zeros(len(class_names), dtype=int); rows = []
    for path in image_paths(images_dir):
        digest = hashlib.sha256(path.read_bytes()).hexdigest()
        if digest in seen_hashes and seen_hashes[digest] != split:
            raise ValueError('Identical image found across splits: ' + path.name)
        seen_hashes[digest] = split
        with Image.open(path) as img: target = read_annotations(path, labels_dir, *img.size)
        for cid in target['labels'].tolist(): counts[cid] += 1
        label_hash = hashlib.sha256((labels_dir / (path.stem+'.txt')).read_bytes()).hexdigest()
        rows.append({'image_sha256': digest, 'label_sha256': label_hash})
    if any(counts == 0): raise ValueError(f"{split} has no examples of: {[class_names[i] for i in np.where(counts==0)[0]]}")
    # Filenames differ across exports; content-based fingerprints remain comparable.
    rows.sort(key=lambda r: (r['image_sha256'],r['label_sha256']))
    manifest[split] = {'images':len(rows),'instances':counts.tolist(),'content':rows}
    print(split, 'images:',len(rows), 'instances:',dict(zip(class_names,counts.tolist())))
fingerprint.update(json.dumps({'classes':class_names,'splits':manifest},sort_keys=True).encode())
DATASET_FINGERPRINT = fingerprint.hexdigest()
META = {'workspace':WORKSPACE,'project':PROJECT,'version':DATASET_VERSION,
        'class_names':class_names,'dataset_fingerprint':DATASET_FINGERPRINT,'seed':SEED,
        'model':MODEL_NAME,'hardware':torch.cuda.get_device_name(0) if device.type=='cuda' else 'cpu',
        'augmentation':'Roboflow version transformations only; additional random training augmentation disabled'}
(RUN_DIR/'dataset_manifest.json').write_text(json.dumps(manifest,indent=2))
(RUN_DIR/'run_metadata.json').write_text(json.dumps(META,indent=2))
versions = {}
for name in ('torch','torchvision','roboflow','numpy','Pillow','ultralytics','torchmetrics','effdet','timm','pycocotools'):
    try: versions[name] = importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError: pass
(RUN_DIR/'package_versions.json').write_text(json.dumps(versions,indent=2))
print('Classes:',class_names,'Output:',RUN_DIR)
print('Exact duplicates checked. Related photos must also have been grouped by survey/session before export.')

def letterbox(img, size):
    w,h=img.size; ratio=min(size/w,size/h)
    nw,nh=max(1,round(w*ratio)),max(1,round(h*ratio))
    left,top=(size-nw)//2,(size-nh)//2
    canvas=Image.new('RGB',(size,size),(114,114,114))
    canvas.paste(img.resize((nw,nh),Image.Resampling.BILINEAR),(left,top))
    return canvas,(nw/w,nh/h,left,top)

## Step 4: Train YOLOv8n

100 maximum epochs, batch 8, input 640, validation early stopping (patience 20). Roboflow training augmentations are used; additional random YOLO transforms are explicitly disabled for comparison.


In [ ]:
from ultralytics import YOLO
model=YOLO('yolov8n.pt')
# Roboflow augmentation is already in the exported train split. Disable additional
# random YOLO augmentation for this controlled three-model experiment.
META['training']={'epochs':100,'batch':8,'imgsz':640,'patience':20,'optimizer':'SGD','pretraining':'COCO detection'}
(RUN_DIR/'run_metadata.json').write_text(json.dumps(META,indent=2))
results=model.train(data=str(dataset_path/'data.yaml'),epochs=100,imgsz=640,batch=8,
    patience=20,device=0 if device.type=='cuda' else 'cpu',project=str(RUN_DIR),name='training',
    seed=SEED,save=True,save_period=1,optimizer='SGD',
    mosaic=0.,mixup=0.,copy_paste=0.,degrees=0.,translate=0.,scale=0.,shear=0.,
    perspective=0.,flipud=0.,fliplr=0.,hsv_h=0.,hsv_s=0.,hsv_v=0.)
model_dir=Path(model.trainer.save_dir)
best_model_path=model_dir/'weights/best.pt'
assert best_model_path.is_file(), 'Training did not produce best.pt'
model=YOLO(str(best_model_path))

## Step 5: Evaluate Model Performance

In [ ]:
metrics=model.val(data=str(dataset_path/'data.yaml'),split=EVAL_SPLIT,conf=.001,max_det=100)
print('Split:',EVAL_SPLIT,'mAP50:',metrics.box.map50,'mAP50-95:',metrics.box.map)
for cid,ap in zip(metrics.box.ap_class_index,metrics.box.maps[metrics.box.ap_class_index]):
    print(model.names[int(cid)],float(ap))

## Step 6: Test on Sample Images

In [ ]:
for path in image_paths(split_dirs[EVAL_SPLIT][0])[:3]:
    predictions=model.predict(str(path),conf=.25,max_det=100)
    predictions[0].show()

## Step 7: Save and Export Model

Save the trained model in multiple formats for deployment

In [ ]:
import shutil
model_path=CHECKPOINT_DIR/'yolov8_coral_best.pt'
shutil.copy2(best_model_path,model_path)
model_path.with_suffix('.metadata.json').write_text(json.dumps(META,indent=2))
print('Best weights:',model_path,'Training run:',model_dir)
# Export only after evaluation. PyTorch weights are sufficient for comparison.
EXPORT_ONNX=False
if EXPORT_ONNX:model.export(format='onnx',imgsz=640)

## Step 8: Download Trained Model

Download the trained model to your local machine

In [ ]:
# Files are already persisted on Drive when USE_DRIVE=True.
import shutil
archive=shutil.make_archive('/content/'+MODEL_NAME+'_run','zip',RUN_DIR)
from google.colab import files
files.download(archive)
print('Published checkpoint:',model_path)

## Ready for your new experiment

No new results are claimed. Run cells top-to-bottom with the reviewed three-class version. Previous outputs were backed up before correction.